# Step 03 — Archived Fold Learning Curves & Manifest Verification (READ-ONLY)

**Purpose:** retrieve original five YOLO11s `results.csv` learning-curve logs, the `5fold_manifest.csv`, and archived four-detector cross-validation CSV files. This notebook **does not train, re-evaluate, edit labels or checkpoints, or change any source files**.

**Scientific scope:** archived fold metrics and split counts can support Reviewer 2 comments 3, 4, 5, 6 and 8. Loss/validation curves provide descriptive diagnostics, **not proof** that the 60-epoch final fit was optimal. Metrics from different evaluation frameworks are **not** presumed harmonized.

The expected source is `MyDrive/Pediatric_Caries_AI/cross_validation/`. Run all cells. Upload `Step_03_Archived_Curves_and_Manifest.zip` when finished.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, csv, json, hashlib, zipfile, statistics
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path('/content/drive/MyDrive/Pediatric_Caries_AI')
CV = PROJECT / 'cross_validation'
OUT = PROJECT / 'reviewer_revision' / 'Step_03_Archived_Statistical_Evidence'
OUT.mkdir(parents=True, exist_ok=True)
print('Project exists:', PROJECT.is_dir())
print('CV directory exists:', CV.is_dir())
assert CV.is_dir(), f'Missing CV directory: {CV}'
print('Output:', OUT)


In [ ]:
# Archive source fold manifest, verify 70 unique images and fold counts.
manifest_path = CV / '5fold_manifest.csv'
assert manifest_path.exists(), f'Please check archived manifest: {manifest_path}'
df = pd.read_csv(manifest_path)
required = {'image_name', 'has_caries', 'caries_boxes', 'fold'}
assert required.issubset(df.columns), f'Missing columns: {required-set(df.columns)}'
assert len(df) == 70, f'Expected 70 development images; found {len(df)}'
assert sorted(df['fold'].unique().tolist()) == [1,2,3,4,5]
assert df['image_name'].astype(str).str.replace('\\','/',regex=False).str.split('/').str[-1].nunique() == 70, 'Duplicate basenames detected; inspect before assuming independent images'
assert int(df['caries_boxes'].sum()) == 206
rows = []
for fold in range(1,6):
    v = df.loc[df['fold'] == fold]
    t = df.loc[df['fold'] != fold]
    rows.append({'fold':fold,'train_images':len(t),'train_positive':int((t['has_caries']==1).sum()),'train_negative':int((t['has_caries']==0).sum()),'train_boxes':int(t['caries_boxes'].sum()),'validation_images':len(v),'validation_positive':int((v['has_caries']==1).sum()),'validation_negative':int((v['has_caries']==0).sum()),'validation_boxes':int(v['caries_boxes'].sum())})
fold_table = pd.DataFrame(rows)
print(fold_table.to_string(index=False))
fold_table.to_csv(OUT / 'fold_distribution_verified_from_manifest.csv',index=False)
shutil.copy2(manifest_path, OUT / '5fold_manifest_ARCHIVED_COPY.csv')
expected_boxes = [50,48,31,49,28]
assert fold_table['validation_boxes'].tolist() == expected_boxes, 'Mismatch versus original notebook logs. Do NOT silently overwrite reported counts.'
print('PASS: manifest matches original archived fold counts.')


In [ ]:
# Retrieve archived detector metrics CSVs (no re-training)
files = {'YOLO11s':'YOLO11s_results.csv','RT-DETR-L':'RTDETR_results.csv','Faster R-CNN':'FasterRCNN_results.csv','RetinaNet':'RetinaNet_results.csv'}
archive_status = []
for model,filename in files.items():
    src = CV / filename
    if not src.exists():
        print('MISSING:',src)
        archive_status.append({'model':model,'source':str(src),'status':'Missing'})
        continue
    d = pd.read_csv(src)
    assert {'fold','mAP50','mAP50_95'}.issubset(d.columns)
    assert sorted(d['fold'].astype(int).tolist()) == [1,2,3,4,5], f'Duplicate/missing folds: {model}'
    shutil.copy2(src,OUT / f'ARCHIVED_{filename}')
    archive_status.append({'model':model,'source':str(src),'status':'Copied; 5 folds'})
    print(model,'mean mAP50:',float(d['mAP50'].mean()),'mean mAP50:95:',float(d['mAP50_95'].mean()))
pd.DataFrame(archive_status).to_csv(OUT / 'archive_status.csv',index=False)


In [ ]:
# Original YOLO11s fold training histories, fold-specific learning curves.
# Only reads results.csv and plots original logged metrics; does not simulate missing epochs.
rows=[];missing=[];histories={}
for fold in range(1,6):
    src = CV / 'models' / 'YOLO11s' / f'fold_{fold}' / 'results.csv'
    if not src.exists():
        missing.append(str(src));continue
    h = pd.read_csv(src);h.columns=[str(c).strip() for c in h.columns]
    assert 'metrics/mAP50-95(B)' in h.columns and 'epoch' in h.columns, f'Unexpected training log columns: {src}'
    assert len(h)>=20, f'Unexpectedly short training history: {src}'
    best_idx = h['metrics/mAP50-95(B)'].astype(float).idxmax()
    best_epoch=int(h.loc[best_idx,'epoch'])+1
    if best_epoch != {1:74,2:66,3:55,4:60,5:40}[fold]:
        raise ValueError(f'Fold {fold} best epoch {best_epoch} differs from saved manuscript/notebook log. Investigate original file versions.')
    rows.append({'fold':fold,'epochs_recorded':len(h),'best_epoch':best_epoch,'best_val_mAP50_95':float(h.loc[best_idx,'metrics/mAP50-95(B)']),'results_csv':str(src)})
    histories[fold]=h
    shutil.copy2(src, OUT / f'YOLO_fold{fold}_results_ARCHIVED.csv')
summary=pd.DataFrame(rows);summary.to_csv(OUT / 'fold_learning_curve_summary.csv',index=False)
print(summary.to_string(index=False))
print('Missing fold histories:',missing)
assert len(histories)==5, 'All five archived YOLO histories are required to close reviewer learning-curve request.'

# Each figure is a standalone plot, not a simulated model fit.
plot_keys=[('metrics/mAP50-95(B)','Validation mAP@0.50:0.95','LearningCurves_Validation_mAP5095.png'),('metrics/mAP50(B)','Validation mAP@0.50','LearningCurves_Validation_mAP50.png'),('train/box_loss','Training box loss','LearningCurves_Training_BoxLoss.png'),('val/box_loss','Validation box loss','LearningCurves_Validation_BoxLoss.png')]
for key,ylab,filename in plot_keys:
    if not all(key in h.columns for h in histories.values()):
        print('Skipped unavailable column:',key);continue
    plt.figure(figsize=(9.2,5.4))
    for fold,h in histories.items():
        plt.plot(h['epoch'].astype(float)+1,h[key].astype(float),label=f'Fold {fold}',linewidth=1.5)
    plt.xlabel('Epoch');plt.ylabel(ylab);plt.title('YOLO11s — archived five-fold histories')
    plt.grid(alpha=.2);plt.legend(ncol=5,fontsize=9)
    plt.tight_layout();plt.savefig(OUT / filename,dpi=210,bbox_inches='tight');plt.close()

(OUT/'readme_scientific_limitations.txt').write_text('Read-only training-log export; original dataset and model weights not changed. Selection of 60 epochs was median of fold-specific best validation mAP50:95 epochs and is not proven optimal. These curves do not remove confounding among model pipelines. Results should not be pooled with later checkpoint snapshots without provenance audit.',encoding='utf8')


In [ ]:
# Bundle all outputs for manuscript revision.
zip_path = OUT.parent / 'Step_03_Archived_Curves_and_Manifest.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print('READY:',zip_path)
print('Files:', len(list(OUT.iterdir())))
from google.colab import files
files.download(str(zip_path))
